# Tree Models
Random forest (RF) and gradient boosted regression trees (GBRT)

## Panel R²oos (monthly, all stocks) 

In [2]:
import pandas as pd
import numpy as np
import gc
import os
import shutil
import warnings
from pathlib import Path
import xgboost as xgb
from sklearn.metrics import mean_squared_error
from itertools import product
from tqdm.auto import tqdm

In [3]:
# ==========================================
# 1. DATA UPLOAD / LOAD PANEL
# ==========================================
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    drive = None
    IN_COLAB = False


DRIVE_FILE = Path('/content/drive/MyDrive/Replication Paper/Data/gkx2020_panel_trimmed.parquet')
LOCAL_COLAB_DATA_DIR = Path('/content/data')
SAVE_OUTPUTS_TO_DRIVE = True
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/Replication Paper/Data/Results/tree_results')

# 2. Load the data
print("Loading data...")

if IN_COLAB:
    drive.mount('/content/drive')
    DATA_DIR = LOCAL_COLAB_DATA_DIR
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    if not DRIVE_FILE.exists():
        raise FileNotFoundError(f"File not found in Google Drive: {DRIVE_FILE}")

    LOCAL_FILE = DATA_DIR / DRIVE_FILE.name
    if not LOCAL_FILE.exists():
        print("Copying file from Google Drive to local Colab runtime...")
        shutil.copy2(DRIVE_FILE, LOCAL_FILE)
        print("Copy complete.")
    else:
        print("Local copy already exists. Skipping copy.")

    if SAVE_OUTPUTS_TO_DRIVE:
        DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
else:
    DATA_DIR = Path('data')
    LOCAL_FILE = next((path for path in LOCAL_FALLBACK_FILES if path.exists()), None)
    if LOCAL_FILE is None:
        searched = ', '.join(str(path) for path in LOCAL_FALLBACK_FILES)
        raise FileNotFoundError(f"Could not find a local panel parquet. Searched: {searched}")

print(f"Reading panel from: {LOCAL_FILE}")
df = pd.read_parquet(LOCAL_FILE)

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
panel_memory_gb = df.memory_usage(deep=True).sum() / 1024**3
panel_dates = df['date'].dt.to_period('M')

print(f"Panel shape: {df.shape}")
print(f"Date range: {panel_dates.min()} to {panel_dates.max()}")
print(f"Unique stocks: {df['permno'].nunique():,}")
print(f"Memory usage: {panel_memory_gb:.2f} GB")

# Define features
cols_to_drop = ['permno', 'date', 'exchcd', 'ret_excess', 'split', 'year', 'me', 'ret', 'prc']
feature_cols = [c for c in df.columns if c not in cols_to_drop]

Mounted at /content/drive
Copying file from Google Drive to local Colab runtime...
Copy complete.
Reading panel from: /content/data/gkx2020_panel_trimmed.parquet
Panel shape: (3305648, 284)
Date range: 1957-03 to 2021-12
Unique stocks: 25,770
Memory usage: 2.74 GB


In [ ]:
# ==========================================
# 2. MODEL CONFIGURATION, TRAINING, AND OUTPUTS
# ==========================================
def detect_cuda():
    """Detect a CUDA GPU without making PyTorch a hard dependency."""
    try:
        import torch
        if torch.cuda.is_available():
            return True, torch.cuda.get_device_name(0)
    except Exception:
        pass

    if Path('/proc/driver/nvidia/version').exists():
        return True, 'NVIDIA GPU detected'

    return False, 'not detected'


CUDA_AVAILABLE, CUDA_DEVICE_NAME = detect_cuda()
XGB_USE_CUDA = CUDA_AVAILABLE

# GBRT debugging defaults. Keep diagnostic mode on for the next short run;
# switch it off after the GBRT diagnostics look healthy.
GBRT_DIAGNOSTIC_MODE = True
GBRT_DIAGNOSTIC_YEARS = [1987, 1988, 1989, 1990]
RUN_RF_IN_DIAGNOSTIC_MODE = False
GBRT_MODEL_NAME = 'GBRT_GKX_HUBER'
GBRT_PRED_COL = 'gbrt_predicted'
GBRT_MAX_TREES = 1000



def xgb_major_version() -> int:
    major = xgb.__version__.split('.', 1)[0]
    digits = ''.join(ch for ch in major if ch.isdigit())
    return int(digits) if digits else 0


def xgb_tree_runtime_params():
    """Return XGBoost tree runtime parameters, using CUDA when Colab exposes a GPU."""
    if XGB_USE_CUDA:
        if xgb_major_version() >= 2:
            return {'tree_method': 'hist', 'device': 'cuda'}
        return {'tree_method': 'gpu_hist', 'predictor': 'gpu_predictor'}

    return {'tree_method': 'hist'}


def to_xgb_array(data):
    """Return data on the same device as the XGBoost booster to avoid predict fallback warnings."""
    if not XGB_USE_CUDA:
        return data

    try:
        import cupy as cp
    except ImportError as exc:
        raise ImportError(
            "CUDA XGBoost prediction without device-mismatch warnings requires CuPy. "
            "In Colab, install a CUDA-matched CuPy build or switch XGB_USE_CUDA = False."
        ) from exc

    if isinstance(data, (pd.DataFrame, pd.Series)):
        array = data.to_numpy(dtype=np.float32, copy=False)
    else:
        array = np.asarray(data, dtype=np.float32)

    return cp.asarray(np.ascontiguousarray(array))


def xgb_predict(model, X):
    """Predict with XGBoost and return a NumPy array for downstream pandas/sklearn code."""
    preds = model.predict(X)

    if XGB_USE_CUDA:
        import cupy as cp
        if isinstance(preds, cp.ndarray):
            return cp.asnumpy(preds)

    return np.asarray(preds)


def clear_xgb_device_cache():
    """Release cached CuPy blocks between annual refits when CUDA is active."""
    if not XGB_USE_CUDA:
        return

    try:
        import cupy as cp
    except ImportError:
        return

    cp.get_default_memory_pool().free_all_blocks()


def max_features_to_colsample_bynode(max_features, n_features):
    """Map sklearn-style RF max_features to XGBoost's colsample_bynode fraction."""
    min_fraction = 1.0 / n_features

    if max_features is None:
        return 1.0
    if isinstance(max_features, str):
        if max_features == 'sqrt':
            return max(min_fraction, 1.0 / np.sqrt(n_features))
        if max_features == 'log2':
            return max(min_fraction, np.log2(n_features) / n_features)
        raise ValueError(f"Unsupported max_features value for XGBoost RF: {max_features!r}")
    if isinstance(max_features, (int, np.integer)):
        return min(1.0, max(min_fraction, float(max_features) / n_features))
    if isinstance(max_features, (float, np.floating)):
        return min(1.0, max(min_fraction, float(max_features)))

    raise TypeError(f"Unsupported max_features type: {type(max_features)!r}")


def make_rf_model(params, n_estimators, n_features):
    """Create an XGBoost random forest, using CUDA when available."""
    rf_params = dict(params)
    max_features = rf_params.pop('max_features', None)
    rf_params['colsample_bynode'] = max_features_to_colsample_bynode(max_features, n_features)

    model_params = {
        'objective': 'reg:squarederror',
        'n_estimators': n_estimators,
        'learning_rate': 1.0,
        'subsample': 0.8,
        'random_state': 10,
        'n_jobs': -1,
        **xgb_tree_runtime_params(),
        **rf_params,
    }

    return xgb.XGBRFRegressor(**model_params)


def as_numpy_array(data):
    """Return a NumPy array from pandas, NumPy, or CuPy inputs."""
    if XGB_USE_CUDA:
        try:
            import cupy as cp
            if isinstance(data, cp.ndarray):
                return cp.asnumpy(data)
        except ImportError:
            pass
    if isinstance(data, (pd.Series, pd.DataFrame)):
        return data.to_numpy()
    return np.asarray(data)


def make_xgb_dmatrix(X, y=None):
    """Create a DMatrix with stable feature names for native XGBoost GBRT."""
    feature_names = list(X.columns) if hasattr(X, 'columns') else feature_cols
    if y is None:
        return xgb.DMatrix(X, feature_names=feature_names)
    return xgb.DMatrix(X, label=as_numpy_array(y).astype(np.float32, copy=False), feature_names=feature_names)


def calculate_gkx_huber_xi(y):
    """GKX Table A.5: Huber threshold xi set to the 99.9% absolute-return quantile."""
    y_arr = np.abs(as_numpy_array(y).astype(np.float64, copy=False))
    xi = float(np.nanquantile(y_arr, GBRT_HUBER_QUANTILE))
    return max(xi, np.finfo(np.float32).eps)


def gkx_huber_loss(y_true, y_pred, xi):
    """Piecewise Huber loss used for GBRT+H validation selection."""
    residual = as_numpy_array(y_true).astype(np.float64, copy=False) - as_numpy_array(y_pred).astype(np.float64, copy=False)
    abs_residual = np.abs(residual)
    loss = np.where(abs_residual <= xi, residual ** 2, 2 * xi * abs_residual - xi ** 2)
    return float(np.nanmean(loss))


def make_gkx_huber_objective(xi):
    """Custom objective for the GKX piecewise Huber loss."""
    def objective(preds, dtrain):
        y = dtrain.get_label()
        residual = preds - y
        abs_residual = np.abs(residual)
        grad = np.where(abs_residual <= xi, 2.0 * residual, 2.0 * xi * np.sign(residual))
        # XGBoost is Newton-style, while paper-style GBRT fits first-order
        # pseudo-residuals. A constant Hessian makes leaf updates average the
        # clipped Huber gradients instead of exploding in the zero-Hessian tail.
        hess = np.full_like(residual, GBRT_HUBER_HESSIAN_VALUE, dtype=np.float64)
        return grad, hess
    return objective


def make_gkx_huber_metric(xi):
    """Validation metric used to choose the number of boosted trees B."""
    def metric(preds, dtrain):
        return 'gkx_huber', gkx_huber_loss(dtrain.get_label(), preds, xi)
    return metric


def make_gbrt_train_params(params):
    """Native XGBoost parameters chosen to mimic GKX shallow GBRT+H."""
    return {
        'objective': 'reg:squarederror',
        'max_depth': params['max_depth'],
        'eta': params['learning_rate'],
        'subsample': GBRT_SUBSAMPLE,
        'colsample_bytree': GBRT_COLSAMPLE_BYTREE,
        'colsample_bynode': GBRT_COLSAMPLE_BYNODE,
        'base_score': 0.0,
        'seed': 10,
        'nthread': -1,
        'disable_default_eval_metric': 1,
        **xgb_tree_runtime_params(),
    }


def train_gkx_gbrt(params, dtrain, xi, num_boost_round, evals=None):
    """Train GKX-style GBRT+H with a fixed boosting budget."""
    import inspect

    evals_result = {}
    train_kwargs = {
        'params': make_gbrt_train_params(params),
        'dtrain': dtrain,
        'num_boost_round': num_boost_round,
        'obj': make_gkx_huber_objective(xi),
        'evals': evals or [],
        'evals_result': evals_result,
        'verbose_eval': False,
    }
    metric = make_gkx_huber_metric(xi)
    if 'custom_metric' in inspect.signature(xgb.train).parameters:
        train_kwargs['custom_metric'] = metric
    else:
        train_kwargs['feval'] = metric

    booster = xgb.train(**train_kwargs)
    return booster, evals_result


print(f"Running in Colab: {IN_COLAB}")
print(f"CUDA available for XGBoost: {CUDA_AVAILABLE} ({CUDA_DEVICE_NAME})")
print(f"XGBoost version: {xgb.__version__}")
print("Note: RF and GBRT both use XGBoost; both train on CUDA when available.")

# ==========================================
# 1. SUPERCOMPUTER MASTER TOGGLE
# ==========================================
# Set to False for local testing. Set to True for the final run.
SUPERCOMPUTER_MODE = True

if SUPERCOMPUTER_MODE:
    print("WARNING: Supercomputer mode enabled. This will take a long time.")
    TEST_YEARS = range(1987, 2022)  # Full GKX-style test period, inclusive through 2021
    VAL_WINDOW_YEARS = 12           # The full GKX validation window
    TUNE_ESTIMATORS = 300           # RF trees during tuning
    FINAL_ESTIMATORS = 300          # RF trees for final refit

    # GKX Appendix A grids (exact match)
    rf_grid = {
        'max_depth': [1, 2, 3, 4, 5, 6],
        'max_features': [3, 5, 10, 20, 30, 50]
    }
    gbrt_grid = {
        'max_depth': [1, 2],
        'learning_rate': [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
        # GKX chooses the number of trees B from 1..1000 on the validation path.
    }

else:
    print("Local Test Mode enabled. Using restricted parameters.")
    TEST_YEARS = [1987, 1988]       # Predict just 2 years
    VAL_WINDOW_YEARS = 2            # Only look back 2 years for validation
    TUNE_ESTIMATORS = 50            # Build shallow RF models to speed up Grid Search
    FINAL_ESTIMATORS = 300

    # The "Speed-Test" Grid Search (Bare minimum to ensure code works)
    rf_grid = {
        'max_depth': [3, 4],
        'max_features': [10]
    }
    gbrt_grid = {
        'max_depth': [1, 2],
        'learning_rate': [0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
    }
# ==========================================

RUN_RANDOM_FOREST = True
if GBRT_DIAGNOSTIC_MODE:
    print("GBRT diagnostic mode enabled. Running only selected stress-test years.")
    TEST_YEARS = GBRT_DIAGNOSTIC_YEARS
    RUN_RANDOM_FOREST = RUN_RF_IN_DIAGNOSTIC_MODE
    if not RUN_RANDOM_FOREST:
        print("Skipping RF in diagnostic mode to keep the run fast.")

# 3. Setup Hyperparameter Grids (Variables now fed from Toggle block)
def make_grid(param_dict):
    keys = param_dict.keys()
    vals = param_dict.values()
    return [dict(zip(keys, v)) for v in product(*vals)]

rf_params_list = make_grid(rf_grid)
gbrt_params_list = make_grid(gbrt_grid)

all_predictions = []
RUN_LABEL = 'GBRT_DIAGNOSTIC' if GBRT_DIAGNOSTIC_MODE else 'TEST'
YEARLY_PREDS_DIR = DATA_DIR / ('yearly_preds_gbrt_diagnostic' if GBRT_DIAGNOSTIC_MODE else 'yearly_preds')
YEARLY_PREDS_DIR.mkdir(parents=True, exist_ok=True)
FINAL_PREDS_FILE = DATA_DIR / ('GBRT_diagnostic_recursive_predictions.parquet' if GBRT_DIAGNOSTIC_MODE else 'TEST_recursive_predictions.parquet')

# 4. GKX OOS R-Squared Formula
def calculate_oos_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denominator = np.sum(y_true ** 2)  # Baseline is predicting zero
    if denominator == 0:
        return np.nan
    numerator = np.sum((y_true - y_pred) ** 2)
    return 1 - (numerator / denominator)


gbrt_diagnostics = []
gbrt_test_diagnostics = []

# 5. THE RECURSIVE REFITTING LOOP
print(f"Starting execution for test years: {TEST_YEARS}")
for test_year in tqdm(TEST_YEARS, desc="Years Processed"):

    # Define time windows for this specific loop
    val_end_year = test_year - 1
    val_start_year = test_year - VAL_WINDOW_YEARS

    # Create masks
    train_mask = df['year'] < val_start_year
    val_mask = (df['year'] >= val_start_year) & (df['year'] <= val_end_year)
    test_mask = df['year'] == test_year

    if test_mask.sum() == 0:
        continue

    # Extract sets. Convert to float32 to save 50% RAM instantly
    X_train = df.loc[train_mask, feature_cols].fillna(0).astype('float32')
    y_train = df.loc[train_mask, 'ret_excess'].astype('float32')

    X_val = df.loc[val_mask, feature_cols].fillna(0).astype('float32')
    y_val = df.loc[val_mask, 'ret_excess'].astype('float32')

    X_test = df.loc[test_mask, feature_cols].fillna(0).astype('float32')
    y_test = df.loc[test_mask, 'ret_excess'].astype('float32')

    # Put XGBoost inputs on the same device as the booster. This avoids the
    # "mismatched devices" fallback warning during validation/test prediction.
    X_train_xgb = to_xgb_array(X_train)
    y_train_xgb = to_xgb_array(y_train)
    X_val_xgb = to_xgb_array(X_val)
    y_val_xgb = to_xgb_array(y_val)
    X_test_xgb = to_xgb_array(X_test)
    dtrain_gbrt = make_xgb_dmatrix(X_train, y_train)
    dval_gbrt = make_xgb_dmatrix(X_val, y_val)
    dtest_gbrt = make_xgb_dmatrix(X_test)

    # --- A. HYPERPARAMETER TUNING (Validation Set) ---
    best_rf_val_mse = float('inf')
    best_rf_params = None

    if RUN_RANDOM_FOREST:
        print(f"\n[{test_year}] Tuning Random Forest...")
        for params in rf_params_list:
            model = make_rf_model(params, TUNE_ESTIMATORS, len(feature_cols))
            model.fit(X_train_xgb, y_train_xgb)
            val_preds = xgb_predict(model, X_val_xgb)
            mse = mean_squared_error(y_val, val_preds)

            if mse < best_rf_val_mse:
                best_rf_val_mse = mse
                best_rf_params = params

        print(f"[{test_year}] Best RF Params: {best_rf_params}")
    else:
        print(f"\n[{test_year}] Skipping Random Forest tuning.")

    print(f"[{test_year}] Tuning {GBRT_MODEL_NAME}...")
    gbrt_huber_xi = calculate_gkx_huber_xi(y_train)
    best_gbrt_val_loss = float('inf')
    best_gbrt_val_mse = float('inf')
    best_gbrt_params = None
    best_gbrt_n_estimators = None
    best_gbrt_val_r2 = np.nan
    # ── FIX 1: Keep the best tuning-phase booster ──────────────────
    # Instead of refitting on train+val later, we save the actual model
    # that was trained on the training set and validated properly.
    best_gbrt_model = None

    for params in gbrt_params_list:
        model, eval_history = train_gkx_gbrt(
            params,
            dtrain_gbrt,
            gbrt_huber_xi,
            num_boost_round=GBRT_MAX_TREES,
            evals=[(dval_gbrt, 'validation')]
        )
        val_losses = np.asarray(eval_history['validation']['gkx_huber'], dtype=np.float64)
        selected_n_estimators = int(np.nanargmin(val_losses)) + 1
        val_huber_loss = float(val_losses[selected_n_estimators - 1])
        val_preds = model.predict(dval_gbrt, iteration_range=(0, selected_n_estimators))
        val_mse = mean_squared_error(y_val, val_preds)
        val_r2 = calculate_oos_r2(y_val, val_preds)
        pred_std = float(np.nanstd(val_preds))
        actual_std = float(np.nanstd(y_val))
        pred_actual_std_ratio = pred_std / actual_std if actual_std > 0 else np.nan

        gbrt_diagnostics.append({
            'test_year': test_year,
            'model': GBRT_MODEL_NAME,
            'objective': 'gkx_piecewise_huber',
            'eval_metric': 'gkx_huber',
            **params,
            'huber_xi': gbrt_huber_xi,
            'max_trees': GBRT_MAX_TREES,
            'selected_n_estimators': selected_n_estimators,
            'val_huber_loss': val_huber_loss,
            'val_mse': val_mse,
            'val_r2_oos': val_r2,
            'pred_mean': float(np.nanmean(val_preds)),
            'pred_std': pred_std,
            'actual_std': actual_std,
            'pred_actual_std_ratio': pred_actual_std_ratio,
            'pred_p01': float(np.nanpercentile(val_preds, 1)),
            'pred_p99': float(np.nanpercentile(val_preds, 99)),
        })

        if GBRT_DIAGNOSTIC_MODE:
            print(
                f"[{test_year}] {GBRT_MODEL_NAME} candidate {params} | "
                f"selected_trees={selected_n_estimators}, val_R2={val_r2 * 100:.4f}%, "
                f"val_huber={val_huber_loss:.8f}, pred_std/actual_std={pred_actual_std_ratio:.4f}"
            )

        if val_huber_loss < best_gbrt_val_loss:
            best_gbrt_val_loss = val_huber_loss
            best_gbrt_val_mse = val_mse
            best_gbrt_val_r2 = val_r2
            best_gbrt_params = dict(params)
            best_gbrt_n_estimators = selected_n_estimators
            # ── FIX 1 (continued): save the winning booster ───────────
            best_gbrt_model = model

    if best_gbrt_params is None:
        raise RuntimeError(f"No valid {GBRT_MODEL_NAME} candidate for test year {test_year}.")

    gbrt_year_config = {
        'params': best_gbrt_params,
        'n_estimators': best_gbrt_n_estimators,
        'huber_xi': gbrt_huber_xi,
        'val_huber_loss': best_gbrt_val_loss,
        'val_mse': best_gbrt_val_mse,
        'val_r2_oos': best_gbrt_val_r2,
    }
    print(f"[{test_year}] Best {GBRT_MODEL_NAME} Params: {best_gbrt_params}")
    print(f"[{test_year}] Best {GBRT_MODEL_NAME} n_estimators: {best_gbrt_n_estimators}")

    # --- B. FIT FINAL MODELS ---
    # ── FIX 1 + FIX 2: NO GBRT REFIT ──────────────────────────────
    # GKX Appendix D: "making out-of-sample predictions using the same
    # fitted model."  The paper trains on the training window, selects
    # hyperparameters (including number of trees) on the validation
    # window, then predicts on the test year with that same model.
    #
    # Refitting GBRT on train+val was the critical bug:
    #   • The optimal tree count was calibrated for train→val, not for
    #     the larger train+val dataset, so the refit uses the wrong
    #     amount of regularisation.
    #   • Huber ξ was recalculated on train+val (FIX 2), changing the
    #     loss function between tuning and deployment.
    #
    # RF is still refit on train+val because its tree count (300) is
    # fixed, not tuned — so the refit is harmless and matches standard
    # practice for bagged ensembles.

    # Merge Train and Val together for RF only
    X_train_full = pd.concat([X_train, X_val])
    y_train_full = pd.concat([y_train, y_val])

    # Free up validation GPU arrays before fitting massive final models.
    del X_train_xgb, y_train_xgb, X_val_xgb, y_val_xgb
    del dtrain_gbrt, dval_gbrt
    clear_xgb_device_cache()

    X_train_full_xgb = to_xgb_array(X_train_full)
    y_train_full_xgb = to_xgb_array(y_train_full)

    # Free up CPU split frames before fitting final models.
    del X_train, y_train, X_val, y_val, model
    gc.collect()

    print(f"[{test_year}] Fitting final models and predicting out-of-sample...")
    # Final RF — refit on train+val is fine (fixed tree count, not tuned)
    if RUN_RANDOM_FOREST:
        final_rf = make_rf_model(best_rf_params, FINAL_ESTIMATORS, len(feature_cols))
        final_rf.fit(X_train_full_xgb, y_train_full_xgb)
        rf_test_preds = xgb_predict(final_rf, X_test_xgb)
    else:
        rf_test_preds = np.full(len(y_test), np.nan, dtype=np.float32)

    # ── FIX 1 + FIX 2: Use the tuning-phase model directly ────────
    # Predict on test with the booster that was trained on the training
    # set, using only the first best_gbrt_n_estimators trees (the count
    # that minimised validation Huber loss).  No refit, no ξ change.
    final_gbrt = best_gbrt_model
    gbrt_test_preds = final_gbrt.predict(
        dtest_gbrt,
        iteration_range=(0, best_gbrt_n_estimators)
    )
    test_pred_std = float(np.nanstd(gbrt_test_preds))
    test_actual_std = float(np.nanstd(y_test))
    test_pred_actual_std_ratio = test_pred_std / test_actual_std if test_actual_std > 0 else np.nan
    test_r2 = calculate_oos_r2(y_test, gbrt_test_preds)
    gbrt_test_diagnostics.append({
        'test_year': test_year,
        'model': GBRT_MODEL_NAME,
        'objective': 'gkx_piecewise_huber',
        **gbrt_year_config['params'],
        'selected_n_estimators': gbrt_year_config['n_estimators'],
        'huber_xi': gbrt_year_config['huber_xi'],
        'selected_val_huber_loss': gbrt_year_config['val_huber_loss'],
        'selected_val_r2_oos': gbrt_year_config['val_r2_oos'],
        'test_r2_oos': test_r2,
        'test_pred_mean': float(np.nanmean(gbrt_test_preds)),
        'test_pred_std': test_pred_std,
        'test_actual_std': test_actual_std,
        'test_pred_actual_std_ratio': test_pred_actual_std_ratio,
        'test_pred_p01': float(np.nanpercentile(gbrt_test_preds, 1)),
        'test_pred_p99': float(np.nanpercentile(gbrt_test_preds, 99)),
    })
    if GBRT_DIAGNOSTIC_MODE:
        print(
            f"[{test_year}] {GBRT_MODEL_NAME} OOS | "
            f"test_R2={test_r2 * 100:.4f}%, "
            f"pred_std/actual_std={test_pred_actual_std_ratio:.4f}"
        )

    # --- C. SAVE RESULTS ---
    # UPDATED: Added permno and mvel1 so we can rank stocks later
    yearly_results = pd.DataFrame({
        'permno': df.loc[test_mask, 'permno'],
        'date': df.loc[test_mask, 'date'],
        'mvel1': df.loc[test_mask, 'mvel1'],
        'actual_return': y_test,
        'rf_predicted': rf_test_preds,
        GBRT_PRED_COL: gbrt_test_preds,
    })

    all_predictions.append(yearly_results)
    yearly_pred_file = YEARLY_PREDS_DIR / f"{RUN_LABEL}_preds_{test_year}.parquet"
    yearly_results.to_parquet(yearly_pred_file)

    if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
        yearly_results.to_parquet(DRIVE_OUTPUT_DIR / yearly_pred_file.name)

    # Aggressive garbage collection to prevent memory crashes across loops
    del X_train_full, y_train_full, X_train_full_xgb, y_train_full_xgb, dtest_gbrt
    del X_test, X_test_xgb, y_test, yearly_results
    del best_gbrt_model  # free the kept booster
    gc.collect()
    clear_xgb_device_cache()

# 6. Final Evaluation
print("\nStitching predictions together...")
final_predictions_df = pd.concat(all_predictions, ignore_index=True)
final_predictions_df.to_parquet(FINAL_PREDS_FILE)

gbrt_diagnostics_df = pd.DataFrame(gbrt_diagnostics)
GBRT_DIAGNOSTICS_FILE = DATA_DIR / 'gbrt_tuning_diagnostics.csv'
gbrt_diagnostics_df.to_csv(GBRT_DIAGNOSTICS_FILE, index=False)
print(f"Saved GBRT tuning diagnostics to {GBRT_DIAGNOSTICS_FILE}")

gbrt_test_diagnostics_df = pd.DataFrame(gbrt_test_diagnostics)
GBRT_TEST_DIAGNOSTICS_FILE = DATA_DIR / 'gbrt_test_diagnostics.csv'
gbrt_test_diagnostics_df.to_csv(GBRT_TEST_DIAGNOSTICS_FILE, index=False)
print(f"Saved GBRT test diagnostics to {GBRT_TEST_DIAGNOSTICS_FILE}")

if GBRT_DIAGNOSTIC_MODE:
    diagnostic_cols = [
        'test_year', 'model', 'objective', 'max_depth', 'learning_rate',
        'huber_xi', 'max_trees', 'selected_n_estimators',
        'val_huber_loss', 'val_r2_oos', 'pred_mean', 'pred_std', 'actual_std',
        'pred_actual_std_ratio', 'pred_p01', 'pred_p99'
    ]
    print("\n--- GBRT Tuning Diagnostics ---")
    print(gbrt_diagnostics_df[diagnostic_cols].to_string(index=False))

    test_diagnostic_cols = [
        'test_year', 'model', 'objective', 'max_depth', 'learning_rate',
        'selected_n_estimators', 'huber_xi',
        'selected_val_huber_loss', 'selected_val_r2_oos', 'test_r2_oos',
        'test_pred_mean', 'test_pred_std', 'test_actual_std',
        'test_pred_actual_std_ratio', 'test_pred_p01', 'test_pred_p99'
    ]
    print("\n--- GBRT Selected Model Test Diagnostics ---")
    print(gbrt_test_diagnostics_df[test_diagnostic_cols].to_string(index=False))

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    combined_out_path = DRIVE_OUTPUT_DIR / FINAL_PREDS_FILE.name
    final_predictions_df.to_parquet(combined_out_path, index=False)
    print(f"Saved combined tree-model predictions to {combined_out_path}")
    gbrt_diagnostics_df.to_csv(DRIVE_OUTPUT_DIR / GBRT_DIAGNOSTICS_FILE.name, index=False)
    gbrt_test_diagnostics_df.to_csv(DRIVE_OUTPUT_DIR / GBRT_TEST_DIAGNOSTICS_FILE.name, index=False)

    # ====================================================
    # Save OOS predictions for each tree model
    # ====================================================
    tree_prediction_specs = {}
    if RUN_RANDOM_FOREST:
        tree_prediction_specs["RF"] = "rf_predicted"
    gbrt_output_name = f"{GBRT_MODEL_NAME}_DIAGNOSTIC" if GBRT_DIAGNOSTIC_MODE else GBRT_MODEL_NAME
    tree_prediction_specs[gbrt_output_name] = GBRT_PRED_COL

    for model_name, pred_col in tree_prediction_specs.items():
        out_df = pd.DataFrame({
            "test_year": final_predictions_df["date"].dt.year,
            "date": final_predictions_df["date"],
            "permno": final_predictions_df["permno"],
            "actual": final_predictions_df["actual_return"],
            "predicted": final_predictions_df[pred_col],
        })
        out_path = DRIVE_OUTPUT_DIR / f"{model_name}_predictions.parquet"
        out_df.to_parquet(out_path, index=False)
        print(f"Saved {model_name} predictions to {out_path}")

    print("\nDone! All tree-model predictions saved.")

print("\n==========================================")
if RUN_RANDOM_FOREST:
    true_rf_r2 = calculate_oos_r2(final_predictions_df['actual_return'], final_predictions_df['rf_predicted'])
    print(f"{RUN_LABEL} RUN - Random Forest R2_OOS: {true_rf_r2 * 100:.3f}%")
else:
    print(f"{RUN_LABEL} RUN - Random Forest R2_OOS: skipped in GBRT diagnostic mode")
gbrt_r2 = calculate_oos_r2(final_predictions_df['actual_return'], final_predictions_df[GBRT_PRED_COL])
print(f"{RUN_LABEL} RUN - {GBRT_MODEL_NAME} R2_OOS: {gbrt_r2 * 100:.3f}%")
print("==========================================")

Running in Colab: True
CUDA available for XGBoost: True (NVIDIA A100-SXM4-40GB)
XGBoost version: 3.2.0
Note: RF and GBRT both use XGBoost; both train on CUDA when available.
GBRT diagnostic mode enabled. Running only selected stress-test years.
Skipping RF in diagnostic mode to keep the run fast.
Starting execution for test years: [1987, 1988, 1989, 1990]


Years Processed:   0%|          | 0/4 [00:00<?, ?it/s]


[1987] Skipping Random Forest tuning.
[1987] Tuning GBRT_GKX_HUBER...


# To check data leakage

In [ ]:

print("\n--- QUICK HUNT FOR DATA LEAKAGE ---")
# 1. Grab just the first 50,000 rows so this runs in 3 seconds
sample_df = df.head(50000).fillna(0)

# 2. Separate features and target
X_sample = sample_df[feature_cols]
y_sample = sample_df['ret_excess']

# 3. Train a quick diagnostic XGBoost Random Forest
print("Training diagnostic model...")
diagnostic_rf = make_rf_model({'max_depth': 5, 'max_features': 'sqrt'}, n_estimators=50, n_features=len(feature_cols))
diagnostic_rf.fit(to_xgb_array(X_sample), to_xgb_array(y_sample))

# 4. Print the top 10 features it relied on
importances = pd.Series(diagnostic_rf.feature_importances_, index=feature_cols)
print("\nTop 10 Most Important Features:")
print(importances.sort_values(ascending=False).head(10))

##  Sub sample R²oos (monthly, top and bottom 1000) 

In [ ]:
# ==========================================
# Deliverable 2: Subsample R²oos (Top 1000 & Bottom 1000)
# ==========================================

print("--- DELIVERABLE 2: Subsample R²oos ---")

# 1. Sort predictions by date, and then by market cap (mvel1) in descending order.
# Dropping NaNs in mvel1 ensures we only rank stocks with valid size data.
sorted_preds = final_predictions_df.dropna(subset=['mvel1']).sort_values(
    by=['date', 'mvel1', 'permno'], 
    ascending=[True, False, True] # False for mvel1 (Largest first), True for permno
)

# 2. Extract the Top 1000 (Largest) and Bottom 1000 (Smallest) stocks per month
top_1000_df = sorted_preds.groupby('date').head(1000)
bottom_1000_df = sorted_preds.groupby('date').tail(1000)

print(f"Total observations in Top 1000 subset: {len(top_1000_df):,}")
print(f"Total observations in Bottom 1000 subset: {len(bottom_1000_df):,}\n")

# 3. Calculate R²oos for the Top 1000
# Note: Re-using the 'calculate_oos_r2' function you defined earlier in the notebook.
rf_available = final_predictions_df['rf_predicted'].notna().any()
top_1000_rf_r2 = calculate_oos_r2(top_1000_df['actual_return'], top_1000_df['rf_predicted']) if rf_available else np.nan

# 4. Calculate R²oos for the Bottom 1000
bot_1000_rf_r2 = calculate_oos_r2(bottom_1000_df['actual_return'], bottom_1000_df['rf_predicted']) if rf_available else np.nan
top_1000_gbrt_r2 = calculate_oos_r2(top_1000_df['actual_return'], top_1000_df[GBRT_PRED_COL])
bot_1000_gbrt_r2 = calculate_oos_r2(bottom_1000_df['actual_return'], bottom_1000_df[GBRT_PRED_COL])
gbrt_subset_table = pd.DataFrame([{
    'Model': GBRT_MODEL_NAME,
    'Top 1000 R2_oos (%)': top_1000_gbrt_r2 * 100,
    'Bottom 1000 R2_oos (%)': bot_1000_gbrt_r2 * 100,
}])

# 5. Display the final metrics (Updated Labels)
print("Top 1000 Largest Stocks (by mvel1):")
print(f"  Random Forest R²oos:   {top_1000_rf_r2 * 100:.3f}%" if rf_available else "  Random Forest R²oos:   skipped")
print(f"  {GBRT_MODEL_NAME} R²oos:       {top_1000_gbrt_r2 * 100:.3f}%")
print()

print("Bottom 1000 Smallest Stocks (by mvel1):")
print(f"  Random Forest R²oos:   {bot_1000_rf_r2 * 100:.3f}%" if rf_available else "  Random Forest R²oos:   skipped")
print(f"  {GBRT_MODEL_NAME} R²oos:       {bot_1000_gbrt_r2 * 100:.3f}%")

## Annual R²oos- Table 2


In [ ]:

# 1. Define the R2_oos calculation function
def calc_r2_oos(df, true_col, pred_col):
    return calculate_oos_r2(df[true_col], df[pred_col])

# Make sure the 'year' column exists
final_predictions_df['year'] = final_predictions_df['date'].dt.year
rf_available = final_predictions_df['rf_predicted'].notna().any()

# 2. Group by year and calculate the year-by-year scores
if rf_available:
    annual_r2_rf = final_predictions_df.groupby('year').apply(
        lambda x: calc_r2_oos(x, 'actual_return', 'rf_predicted'),
        include_groups=False
    )

# 3. Take the SIMPLE AVERAGE across all testing years and convert to percentage
avg_annual_rf = annual_r2_rf.mean() * 100 if rf_available else np.nan

# 4. Create the final formatted Table 2 to match the paper
table_rows = []
if rf_available:
    table_rows.append({'Model': 'RF', 'Annual R2_oos (%)': avg_annual_rf})
annual_r2_gbrt = final_predictions_df.groupby('year').apply(
    lambda x: calc_r2_oos(x, 'actual_return', GBRT_PRED_COL),
    include_groups=False
)
table_rows.append({'Model': GBRT_MODEL_NAME, 'Annual R2_oos (%)': annual_r2_gbrt.mean() * 100})
table_2_final = pd.DataFrame(table_rows)

print("--- Table 2: Annual Out-of-Sample Predictive R2 ---")
print(table_2_final.to_string(index=False))

# Monthly prediction error series for DM tests 

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Ensure date is a proper datetime format (if not already)
final_predictions_df['date'] = pd.to_datetime(final_predictions_df['date'])

# 2. Calculate squared errors for each observation
# The benchmark prediction is exactly 0, so its error is just the actual return
se_bench = final_predictions_df['actual_return'] ** 2
rf_available = final_predictions_df['rf_predicted'].notna().any()
if rf_available:
    se_rf = (final_predictions_df['actual_return'] - final_predictions_df['rf_predicted']) ** 2
se_gbrt = (final_predictions_df['actual_return'] - final_predictions_df[GBRT_PRED_COL]) ** 2
final_predictions_df['d_gbrt'] = se_bench - se_gbrt

# 3. Calculate the difference in squared errors (d_it)
# A positive number here means the model beat the benchmark!
if rf_available:
    final_predictions_df['d_rf'] = se_bench - se_rf

# 4. Group by date (month) to get the cross-sectional average difference (d_t)
dm_cols = ['d_gbrt']
if rf_available:
    dm_cols.insert(0, 'd_rf')
monthly_d = final_predictions_df.groupby('date')[dm_cols].mean()

# 5. Define a function to compute the DM statistic with Newey-West standard errors
def calculate_dm_stat(d_t_series, lags=6):
    model = sm.OLS(d_t_series, np.ones(len(d_t_series)))
    results = model.fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    
    # Use .iloc[0] instead of [0] to fix the FutureWarning!
    dm_stat = results.tvalues.iloc[0]
    p_value = results.pvalues.iloc[0]
    
    return dm_stat, p_value

# 6. Calculate DM stats for both models
if rf_available:
    dm_rf, p_rf = calculate_dm_stat(monthly_d['d_rf'].dropna())
dm_gbrt, p_gbrt = calculate_dm_stat(monthly_d['d_gbrt'].dropna())

# 7. Combine into a neat summary table
dm_rows = []
if rf_available:
    dm_rows.append({'Model': 'Random Forest', 'DM Statistic': dm_rf, 'p-value': p_rf})
dm_rows.append({'Model': GBRT_MODEL_NAME, 'DM Statistic': dm_gbrt, 'p-value': p_gbrt})
table_3_summary = pd.DataFrame(dm_rows)

print("Table 3: Diebold-Mariano Tests")
print(table_3_summary.to_string(index=False))
monthly_d.to_csv("data/rf_gbrt_monthly_DM_series.csv")

# Variable importance of top 20 characteristics for RF and GBRT variants

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print("--- DELIVERABLE 5: Figure 4 (Variable Importance) ---")
rf_available_for_importance = RUN_RANDOM_FOREST and 'final_rf' in globals()
importance_dir = Path("data")
importance_dir.mkdir(exist_ok=True)


def make_sklearn_importance_df(model, feature_cols):
    importances = np.asarray(model.feature_importances_, dtype=float)
    total_importance = np.nansum(importances)
    if np.isfinite(total_importance) and total_importance > 0:
        importances = importances / total_importance
    return pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)


def make_booster_importance_df(booster, feature_cols, importance_type='gain'):
    score = booster.get_score(importance_type=importance_type)
    importances = pd.Series(0.0, index=feature_cols, dtype=float)
    for feature, value in score.items():
        if feature in importances.index:
            importances.loc[feature] = value
        elif feature.startswith('f') and feature[1:].isdigit():
            feature_idx = int(feature[1:])
            if feature_idx < len(feature_cols):
                importances.iloc[feature_idx] = value
    total_importance = importances.sum()
    if np.isfinite(total_importance) and total_importance > 0:
        importances = importances / total_importance
    return importances.rename_axis('Feature').reset_index(name='Importance').sort_values(
        by='Importance', ascending=False
    )

# ==========================================
# 1. Random Forest Importance
# ==========================================
if rf_available_for_importance:
    rf_imp_df = make_sklearn_importance_df(final_rf, feature_cols)
    top20_rf = rf_imp_df.head(20)
else:
    print("Random Forest importance skipped because RF was not run in diagnostic mode.")

# ==========================================
# 2. GKX GBRT+H (native XGBoost Booster) Importance
# ==========================================
gbrt_imp_df = None
if 'final_gbrt' in globals():
    gbrt_imp_df = make_booster_importance_df(final_gbrt, feature_cols, importance_type='gain')
    top20_gbrt = gbrt_imp_df.head(20)
else:
    print("GBRT importance skipped because no final GBRT models are currently in memory.")

# ==========================================
# 3. Plotting Figure 4
# ==========================================
plot_items = []
if rf_available_for_importance:
    plot_items.append(('Random Forest', top20_rf, 'steelblue'))
if gbrt_imp_df is not None:
    plot_items.append((GBRT_MODEL_NAME, top20_gbrt, 'seagreen'))

if plot_items:
    fig, axes = plt.subplots(1, len(plot_items), figsize=(8 * len(plot_items), 8), squeeze=False)
    for ax, (model_name, top20_df, color) in zip(axes.ravel(), plot_items):
        ax.barh(top20_df['Feature'][::-1], top20_df['Importance'][::-1], color=color)
        ax.set_title(f'{model_name} - Top 20 Characteristics', fontsize=14)
        ax.set_xlabel('Normalized Importance', fontsize=12)
        ax.grid(axis='x', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

# Save the data to CSV for the final report
if rf_available_for_importance:
    rf_imp_df.to_csv(importance_dir / "rf_feature_importance.csv", index=False)
if gbrt_imp_df is not None:
    gbrt_imp_df.to_csv(importance_dir / f"{GBRT_MODEL_NAME.lower()}_feature_importance.csv", index=False)
    gbrt_imp_df.to_csv(importance_dir / "gbrt_feature_importance.csv", index=False)